In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from collections import Counter
from itertools import combinations
import ast  # For safely evaluating string representation of lists
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
from wordcloud import WordCloud


In [3]:
# ==========================================
# A. SETUP & DATA RECALL
# ==========================================
print("--- Section A: Setup & Data Recall ---")

# Task 1: Load data
try:
    df_comments = pd.read_csv('/kaggle/input/cleans/cleaned_comments.csv')
    df_captions = pd.read_csv('/kaggle/input/cleans/cleaned_captions.csv')
    print("Files loaded successfully.")
except FileNotFoundError:
    print("Error: Files not found. Please run generate_dummy_data.py first.")
    exit()

# Task 2 & 3: Verify and clean 'cleaned_tokens'
# We use ast.literal_eval to convert the string "['a', 'b']" back to a list ['a', 'b']
def parse_tokens(text):
    if pd.isna(text): return []
    try:
        return ast.literal_eval(text)
    except:
        return []

df_comments['cleaned_tokens'] = df_comments['cleaned_tokens'].apply(parse_tokens)
# Drop empty rows
df_comments = df_comments[df_comments['cleaned_tokens'].map(len) > 0]

# Task 4: Top 20 Unigrams
all_tokens = [token for sublist in df_comments['cleaned_tokens'] for token in sublist]
top_20 = Counter(all_tokens).most_common(20)
print(f"Top 5 Unigrams: {top_20[:5]}")

--- Section A: Setup & Data Recall ---
Files loaded successfully.
Top 5 Unigrams: [('zebra', 17), ('beautiful', 9), ('nature', 9), ('migration', 8), ('journey', 7)]


In [4]:
# ==========================================
# B. TRANSACTION CONSTRUCTION
# ==========================================
print("\n--- Section B: Transaction Construction ---")

# Task 6, 7, 8: Create transactions (Baskets)
transactions = []
for tokens in df_comments['cleaned_tokens']:
    # Task 8: Remove duplicates within basket (set) and Task 7: Check length
    unique_tokens = list(set(tokens))
    if len(unique_tokens) >= 3:
        transactions.append(unique_tokens)

print(f"Number of valid transactions (length >= 3): {len(transactions)}")

# Task 11: Histogram of basket lengths
basket_lengths = [len(t) for t in transactions]
plt.figure(figsize=(8, 5))
plt.hist(basket_lengths, bins=range(min(basket_lengths), max(basket_lengths) + 2), edgecolor='black')
plt.title('Task 11: Histogram of Basket Lengths')
plt.xlabel('Number of items in basket')
plt.ylabel('Frequency')
plt.savefig('basket_lengths_histogram.png')
plt.close()

# Task 12: Statistics
print(f"Avg Length: {np.mean(basket_lengths):.2f}")
print(f"Min Length: {np.min(basket_lengths)}")
print(f"Max Length: {np.max(basket_lengths)}")



--- Section B: Transaction Construction ---
Number of valid transactions (length >= 3): 41
Avg Length: 13.27
Min Length: 3
Max Length: 61


In [5]:
# ==========================================
# C. MANUAL PATTERN DISCOVERY
# ==========================================
print("\n--- Section C: Manual Pattern Discovery ---")

# Task 13 & 14: Count pairwise co-occurrences
pair_counts = Counter()
for transaction in transactions:
    # Sort to ensure (A, B) is same as (B, A)
    for pair in combinations(sorted(transaction), 2):
        pair_counts[pair] += 1

# Task 15: Filter count >= 3
frequent_pairs = {k: v for k, v in pair_counts.items() if v >= 3}
sorted_pairs = sorted(frequent_pairs.items(), key=lambda x: x[1], reverse=True)

# Task 16: Plot Top 20 Co-occurring pairs
top_20_pairs = sorted_pairs[:20]
labels = [f"{p[0]}-{p[1]}" for p, count in top_20_pairs]
counts = [count for p, count in top_20_pairs]

plt.figure(figsize=(10, 8))
plt.barh(labels, counts, color='skyblue')
plt.gca().invert_yaxis()
plt.title('Task 16: Top 20 Word Co-occurrences')
plt.savefig('manual_co_occurrence.png')
plt.close()

# Task 18: Network Graph
G = nx.Graph()
# Add edges for top 30 pairs to keep graph readable
for (w1, w2), count in sorted_pairs[:30]:
    G.add_edge(w1, w2, weight=count)

plt.figure(figsize=(10, 8))
pos = nx.spring_layout(G, k=1.5)
nx.draw(G, pos, with_labels=True, node_color='lightgreen', 
        node_size=2000, font_size=10, font_weight='bold')
plt.title('Task 18: Co-occurrence Network Graph')
plt.savefig('network_graph.png')
plt.close()

# Task 20: Export
pd.DataFrame(sorted_pairs, columns=['Pair', 'Count']).to_csv('manual_co_occurrences.csv', index=False)



--- Section C: Manual Pattern Discovery ---


In [6]:
# ==========================================
# D. ALGORITHMIC PATTERN DISCOVERY (APRIORI)
# ==========================================
print("\n--- Section D: Algorithmic Pattern Discovery ---")

# Task 21: One-hot encoding
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

# Task 22 & 23: Run Apriori (Using min_support=0.1 for demonstration)
# Note: In a real sparse dataset, 0.3 might be too high. 
# We use 0.1 here to ensure we get results with the dummy data.
MIN_SUPPORT = 0.1 
print(f"Running Apriori with min_support={MIN_SUPPORT}...")

frequent_itemsets = apriori(df_encoded, min_support=MIN_SUPPORT, use_colnames=True)

# Task 24: Filter by length (add a length column)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(lambda x: len(x))
filtered_itemsets = frequent_itemsets[ (frequent_itemsets['length'] >= 2) & (frequent_itemsets['length'] <= 3) ]

print(f"Found {len(frequent_itemsets)} frequent itemsets.")
print(f"Found {len(filtered_itemsets)} itemsets with length 2 or 3.")

# Task 25: Association Rules
# We use a low threshold here to ensure we generate rules for the lab
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.1)

# Task 26 & 27: Filter Rules
filtered_rules = rules[ (rules['confidence'] >= 0.6) & (rules['lift'] >= 1.2) ]
print(f"Generated {len(rules)} rules. After filtering (conf>=0.6, lift>=1.2): {len(filtered_rules)}")

# Task 28: Support vs Confidence Scatter Plot
plt.figure(figsize=(8, 6))
plt.scatter(rules['support'], rules['confidence'], alpha=0.5)
plt.xlabel('Support')
plt.ylabel('Confidence')
plt.title('Task 28: Support vs Confidence')
plt.savefig('support_vs_confidence.png')
plt.close()

# Task 29 & 30: Save Results
frequent_itemsets.to_csv('frequent_itemsets.csv', index=False)
filtered_rules.to_csv('association_rules.csv', index=False)



--- Section D: Algorithmic Pattern Discovery ---
Running Apriori with min_support=0.1...
Found 14 frequent itemsets.
Found 4 itemsets with length 2 or 3.
Generated 8 rules. After filtering (conf>=0.6, lift>=1.2): 5


In [7]:
# ==========================================
# E. EXPLORATION (Sample Logic)
# ==========================================
print("\n--- Section E: Exploration ---")
print("Experimenting with min_support=0.05...")
# Task 32: Try min_support 0.05
frequent_itemsets_low = apriori(df_encoded, min_support=0.05, use_colnames=True)
print(f"Itemsets at 0.05 support: {len(frequent_itemsets_low)}")

# Task 34: Token length filtering example (preprocessing logic)
# This code block demonstrates how you would re-filter the transactions
filtered_transactions = [[t for t in basket if len(t) >= 4] for basket in transactions]
# You would then re-run the TransactionEncoder and Apriori on `filtered_transactions`


--- Section E: Exploration ---
Experimenting with min_support=0.05...
Itemsets at 0.05 support: 69


In [9]:
#==========================================
# F. VISUALIZATION & INTERPRETATION
# ==========================================
print("\n--- Section F: Visualization ---")

# Task 39: Bar chart of top 10 2-itemsets by support
top_itemsets = filtered_itemsets[filtered_itemsets['length'] == 2].sort_values(by='support', ascending=False).head(10)

plt.figure(figsize=(10, 6))
# Convert frozenset to string for plotting
labels = [str(list(x)) for x in top_itemsets['itemsets']]
plt.barh(labels, top_itemsets['support'], color='salmon')
plt.gca().invert_yaxis()
plt.title('Task 39: Top 10 2-Itemsets by Support')
plt.savefig('top_itemsets_support.png')
plt.close()

# Task 41: Word Cloud
# Flatten all items in valid transactions
text_data = " ".join([word for transaction in transactions for word in transaction])
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text_data)

plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Task 41: Word Cloud of Frequent Items')
plt.savefig('wordcloud.png')
plt.close()

# Task 44: Insight Statements (Automated print based on top rules)
print("\n--- Task 44: Insights ---")
if not filtered_rules.empty:
    top_rules = filtered_rules.sort_values(by='lift', ascending=False).head(3)
    for idx, row in top_rules.iterrows():
        ant = list(row['antecedents'])[0]
        con = list(row['consequents'])[0]
        print(f"Insight: Users who mention '{ant}' are likely to mention '{con}'. "
              f"(Support: {row['support']:.2f}, Conf: {row['confidence']:.2f}, Lift: {row['lift']:.2f})")
else:
    print("No rules met the strict criteria to generate insights.")

print("\n--- Lab 3 Processing Complete. Check output PNGs and CSVs. ---")


--- Section F: Visualization ---

--- Task 44: Insights ---
Insight: Users who mention 'earth' are likely to mention 'bbc'. (Support: 0.12, Conf: 0.83, Lift: 6.83)
Insight: Users who mention 'bbc' are likely to mention 'earth'. (Support: 0.12, Conf: 1.00, Lift: 6.83)
Insight: Users who mention 'animal' are likely to mention 'zebra'. (Support: 0.12, Conf: 0.83, Lift: 2.28)

--- Lab 3 Processing Complete. Check output PNGs and CSVs. ---
